# 🛒 Market Basket Analysis (Apriori)
> **Caso de negocio:** Crisol · Cross-sell en librería online
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

Crisol vende manga, artbooks, figuras coleccionables y útiles de diseño en su tienda online. Hoy las recomendaciones de "productos relacionados" se configuran manualmente por el equipo de e-commerce, sin un análisis sistemático de qué productos se compran juntos.

**Objetivo:** Identificar asociaciones entre productos (reglas "si compra A, también compra B") para alimentar recomendaciones de cross-sell, bundles y ubicación de productos en la tienda.

**KPIs de éxito:**
- Reglas con `lift > 1` (asociación más fuerte que el azar)
- `confidence >= 0.3` para las reglas usadas en recomendaciones
- Al menos 3 reglas de cross-sell accionables para el equipo de e-commerce

**Algoritmo:** Apriori (itemsets frecuentes) + Association Rules (`support`, `confidence`, `lift`)

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q mlxtend plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Aprendizaje no supervisado — reglas de asociación',
    'unidad_analisis': 'ticket_id (boleta de compra) x producto',
    'metricas_clave': [
        'support    → frecuencia con la que aparece un itemset en todas las boletas',
        'confidence → P(consecuente | antecedente)',
        'lift       → cuánto más probable es B dado A, vs. comprar B al azar',
    ],
    'criterios_aceptacion': {
        'lift':       '> 1.0  (asociación real, no azar)',
        'confidence': '>= 0.30',
        'reglas_accionables': '>= 3 para cross-sell',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con estructura realista
# En producción: reemplazar con detalle de boletas (BigQuery / ERP de ventas)
N_TICKETS = 1200

prods = ['Manga shonen', 'Artbook anime', 'Poster coleccionable',
         'Novela grafica', 'Marcadores premium', 'Cuaderno diseno',
         'Stickers kawaii', 'Boligrafo premium', 'Novela light',
         'Figura coleccionable']

# Reglas "ocultas" del DGP: (producto_a, producto_b, probabilidad de co-compra)
hidden_rules = [(0,1,0.65), (0,2,0.40), (3,5,0.35), (1,6,0.50), (7,5,0.42), (4,8,0.38)]

rows = []
for tid in range(1, N_TICKETS+1):
    n_prods = np.random.randint(1, 5)
    basket = set()
    anchor = np.random.randint(0, len(prods))
    basket.add(anchor)
    for a, b, p in hidden_rules:
        if a == anchor and np.random.random() < p:
            basket.add(b)
    while len(basket) < n_prods:
        basket.add(np.random.randint(0, len(prods)))
    for p in basket:
        rows.append({'ticket_id': f'T{tid}', 'producto': prods[p]})

df = pd.DataFrame(rows)

print(f'Líneas de boleta: {df.shape[0]}')
print(f'Boletas (tickets): {df["ticket_id"].nunique()}')
print(f'Productos distintos: {df["producto"].nunique()}')
df.head()

In [ ]:
# Tamaño de canasta y frecuencia de productos
basket_size = df.groupby('ticket_id')['producto'].count()

fig = px.histogram(basket_size, nbins=4,
                    title='Distribución del tamaño de la canasta (productos por boleta)',
                    color_discrete_sequence=['#1a4c8c'])
fig.update_layout(height=350, plot_bgcolor='white', paper_bgcolor='white',
                  showlegend=False, xaxis_title='Productos por boleta', yaxis_title='Boletas')
fig.show()
print(f'Tamaño promedio de canasta: {basket_size.mean():.2f} productos')

In [ ]:
# Frecuencia de cada producto (% de boletas en las que aparece)
freq_producto = (df.groupby('producto')['ticket_id'].nunique() / df['ticket_id'].nunique()).sort_values(ascending=False)

fig = px.bar(x=freq_producto.values, y=freq_producto.index, orientation='h',
             title='Frecuencia de productos (% de boletas)',
             text=[f'{v:.1%}' for v in freq_producto.values],
             color=freq_producto.values, color_continuous_scale='teal')
fig.update_traces(textposition='outside')
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white',
                  showlegend=False, xaxis_title='% de boletas', yaxis_title=None,
                  xaxis_tickformat='.0%')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
# Codificación one-hot de transacciones: una fila por boleta, una columna por producto
transactions = df.groupby('ticket_id')['producto'].apply(list).tolist()

te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_te = pd.DataFrame(te_array, columns=te.columns_)

print(f'Matriz de transacciones: {df_te.shape[0]} boletas x {df_te.shape[1]} productos')
df_te.head()

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Itemsets frecuentes con soporte mínimo de 5%
frequent = apriori(df_te, min_support=0.05, use_colnames=True, verbose=0)
frequent = frequent.sort_values('support', ascending=False)

print(f'Itemsets frecuentes encontrados: {len(frequent)}')
frequent.head(10)

In [ ]:
# Reglas de asociación con confidence mínima de 30% y lift > 1
rules = association_rules(frequent, metric='confidence', min_threshold=0.3)
rules = rules[rules['lift'] >= 1.0].sort_values('lift', ascending=False).reset_index(drop=True)

rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

print(f'Reglas encontradas (lift > 1, confidence >= 0.3): {len(rules)}')
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

## 5️⃣ Fase 5 — Evaluation

In [ ]:
metrics = {
    'n_transacciones': df['ticket_id'].nunique(),
    'n_itemsets_frecuentes': len(frequent),
    'n_reglas': len(rules),
    'lift_max': rules['lift'].max(),
    'confidence_promedio': rules['confidence'].mean(),
}

print('=== MÉTRICAS DEL MODELO ===')
for k, v in metrics.items():
    print(f'  {k:22s}: {v:.3f}' if isinstance(v, float) else f'  {k:22s}: {v}')

estado = '✅ APROBADO' if len(rules) >= 3 else '❌ NECESITA MEJORA (se requieren >= 3 reglas)'
print(f'\nCriterio >= 3 reglas accionables: {estado}')

In [ ]:
# Top reglas por lift
top = rules.head(10).copy()
top['regla'] = top['antecedents'] + ' → ' + top['consequents']

fig = go.Figure(go.Bar(
    x=top['lift'], y=top['regla'], orientation='h',
    marker_color='#1a4c8c',
    text=top['lift'].map('{:.2f}'.format), textposition='outside'
))
fig.add_vline(x=1, line_dash='dash', line_color='gray', annotation_text='lift = 1 (azar)')
fig.update_layout(title='Top reglas de asociación por Lift',
                  height=max(300, len(top)*45), plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0', title='Lift'), yaxis=dict(showgrid=False))
fig.show()

In [ ]:
# Mapa support vs confidence — tamaño = lift
fig = px.scatter(rules, x='support', y='confidence', size='lift', color='lift',
                  hover_data=['antecedents', 'consequents'],
                  title='Reglas de asociación: support vs confidence (tamaño = lift)',
                  color_continuous_scale='teal')
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0', tickformat='.0%'),
                  yaxis=dict(gridcolor='#f0f0f0', tickformat='.0%'))
fig.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Recomendación de cross-sell: mejor producto a sugerir para cada producto ancla
cross_sell = (rules[rules['antecedents'].str.split(', ').apply(len) == 1]
               .sort_values(['antecedents', 'lift'], ascending=[True, False])
               .groupby('antecedents')
               .first()
               .reset_index()[['antecedents', 'consequents', 'confidence', 'lift']])
cross_sell.columns = ['producto', 'recomendar_con', 'confidence', 'lift']

print('Recomendaciones de cross-sell por producto:')
display(cross_sell)

In [ ]:
# Guardar outputs
rules.to_csv('reglas_asociacion_crisol.csv', index=False)
cross_sell.to_csv('cross_sell_crisol.csv', index=False)
print('Archivos reglas_asociacion_crisol.csv y cross_sell_crisol.csv guardados ✓')

import joblib
joblib.dump({'frequent': frequent, 'rules': rules}, 'modelo_basket_crisol.pkl')
print('Modelo modelo_basket_crisol.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
mejor_regla = rules.iloc[0]

print('=== RESUMEN EJECUTIVO ===')
print(f'Boletas analizadas:                 {df["ticket_id"].nunique():,}')
print(f'Reglas de asociación encontradas:   {len(rules)}')
print(f'Recomendaciones de cross-sell:      {len(cross_sell)} productos con sugerencia')
print(f'Regla más fuerte:                   {mejor_regla["antecedents"]} → {mejor_regla["consequents"]}')
print(f'                                     (lift={mejor_regla["lift"]:.2f}, confidence={mejor_regla["confidence"]:.1%})')
print(f'\nArquitectura de producción:')
print('  Detalle de boletas → BigQuery → recalculo semanal de reglas → módulo "frecuentemente comprados juntos"')